<a href="https://colab.research.google.com/github/NabilBADRI/Competition-StanceNakba-2026/blob/main/competition_NLP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import string
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import model_selection, naive_bayes, svm
from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report as creport
from sklearn.model_selection import train_test_split

from multiprocessing import Pool

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_train.csv')

In [ ]:
# Count the occurrences of each stance_label
print(df['stance_label'].value_counts())

# Utiliser Transformers avec BERT arabe

In [ ]:
!pip install transformers torch scikit-learn pandas

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
from sklearn.metrics import classification_report

In [ ]:
# Charger un modèle pré-entraîné pour l'arabe
model_name = "aubmindlab/bert-base-arabertv2"  # Modèle BERT pour l'arabe
# Ou: "UBC-NLP/MARBERT" - Modèle spécifique pour l'arabe

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3  # pro, against, neutral
)

def classify_with_transformers(texts, model, tokenizer):
    """Classifie des textes avec un modèle transformers"""
    predictions = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item()

        # Mapper aux labels (à adapter selon votre encodage)
        if predicted_class == 0:
            predictions.append('pro')
        elif predicted_class == 1:
            predictions.append('against')
        else:
            predictions.append('neutral')

    return predictions

# Tester
sample_texts = df['Sentence'].head(10).tolist()
predictions = classify_with_transformers(sample_texts, model, tokenizer)
print(predictions)

In [ ]:
!pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [ ]:
# Charger modèle pour l'arabe
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Encoder les textes
embeddings = model.encode(df['Sentence'].tolist())

# Diviser en train/test
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, df['stance_label'], test_size=0.2, random_state=42
)

# Entraîner un classifieur
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Évaluer
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
##### best one ,i added validation
import numpy as np
import pandas as pd
import joblib
import pickle
import datetime
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, log_loss, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. LOAD DATA AND INITIALIZE MODEL
# ============================================
print("Loading model and preparing data...")

# Charger modèle pour l'arabe
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Assuming df is your DataFrame with 'Sentence' and 'stance_label' columns
# df = pd.read_csv('your_data.csv')

# Encoder les textes
print("Encoding sentences...")
embeddings = model.encode(df['Sentence'].tolist(), show_progress_bar=True)

# Diviser en train/validation/test (60-20-20 split)
print("Splitting data...")
X_train, X_temp, y_train, y_temp = train_test_split(
    embeddings, df['stance_label'], test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}, Test size: {len(X_test)}")

# ============================================
# 2. CREATE AND TRAIN PIPELINE
# ============================================
print("\n" + "="*50)
print("TRAINING PIPELINE")
print("="*50)

# Create pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Define parameter grid for tuning
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga']
}

# Perform grid search with cross-validation
print("Performing hyperparameter tuning with GridSearchCV...")
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

# Combine train and validation for final training
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

# Fit the grid search
grid_search.fit(X_train_full, y_train_full)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best CV score (F1-macro): {grid_search.best_score_:.3f}")

# Get the best model
best_model = grid_search.best_estimator_

# ============================================
# 3. EVALUATE ON TEST SET
# ============================================
print("\n" + "="*50)
print("TEST SET EVALUATION")
print("="*50)

# Predict on test set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate additional metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.3f}")

# For binary classification
if len(np.unique(y_train_full)) == 2:
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])
    print(f"AUC-ROC: {auc_roc:.3f}")
    test_loss = log_loss(y_test, y_pred_proba)
    print(f"Log Loss: {test_loss:.3f}")
else:
    # For multi-class
    auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
    print(f"AUC-ROC (One-vs-Rest): {auc_roc:.3f}")

# ============================================
# 4. VISUALIZATION AND ERROR ANALYSIS
# ============================================
print("\n" + "="*50)
print("VISUALIZATION AND ERROR ANALYSIS")
print("="*50)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(np.unique(y_test)),
            yticklabels=sorted(np.unique(y_test)))
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Error Analysis
error_indices = np.where(y_pred != y_test.values)[0]
error_rate = len(error_indices) / len(y_test)
print(f"\nError Analysis:")
print(f"Total errors: {len(error_indices)}")
print(f"Error rate: {error_rate:.2%}")

if len(error_indices) > 0:
    # Create error dataframe
    test_sentences = df.loc[y_test.index, 'Sentence'].reset_index(drop=True)
    error_df = pd.DataFrame({
        'Sentence': test_sentences.iloc[error_indices],
        'True_Label': y_test.values[error_indices],
        'Predicted_Label': y_pred[error_indices],
        'Confidence': np.max(y_pred_proba[error_indices], axis=1)
    })

    print("\nSample of misclassified sentences:")
    print(error_df.head(10).to_string())

    # Save errors to CSV
    error_df.to_csv('misclassified_examples.csv', index=False, encoding='utf-8-sig')

# Feature Importance (for logistic regression)
if hasattr(best_model.named_steps['classifier'], 'coef_'):
    print("\nTop 10 Most Important Features:")
    feature_importance = pd.DataFrame({
        'feature': range(len(best_model.named_steps['classifier'].coef_[0])),
        'importance': abs(best_model.named_steps['classifier'].coef_[0])
    }).sort_values('importance', ascending=False)
    print(feature_importance.head(10).to_string())

# ============================================
# 5. SAVE MODELS AND METADATA
# ============================================
print("\n" + "="*50)
print("SAVING MODELS AND METADATA")
print("="*50)

# Create unique version identifier
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
data_hash = hashlib.md5(df['Sentence'].astype(str).sum().encode()).hexdigest()[:8]
version_str = f"model_v{timestamp}_{data_hash}"

# Create directory for saving
import os
os.makedirs('saved_models', exist_ok=True)

# 5.1 Save the Sentence Transformer model
print("Saving Sentence Transformer model...")
model.save(f'saved_models/{version_str}_sentence_transformer')

# 5.2 Save the sklearn classifier pipeline
print("Saving classifier pipeline...")
joblib.dump(best_model, f'saved_models/{version_str}_classifier.pkl')

# 5.3 Save complete pipeline metadata
print("Saving metadata...")
metadata = {
    'version': version_str,
    'timestamp': datetime.datetime.now().isoformat(),
    'model_name': 'paraphrase-multilingual-MiniLM-L12-v2',
    'data_info': {
        'total_samples': len(df),
        'train_size': len(X_train_full),
        'test_size': len(X_test),
        'num_classes': len(np.unique(y_train_full)),
        'class_distribution': dict(pd.Series(y_train_full).value_counts().sort_index())
    },
    'hyperparameters': grid_search.best_params_,
    'performance_metrics': {
        'test_accuracy': float(accuracy),
        'test_f1_macro': float(f1_score(y_test, y_pred, average='macro')),
        'test_f1_weighted': float(f1_score(y_test, y_pred, average='weighted')),
        'cv_best_score': float(grid_search.best_score_)
    },
    'classification_report': classification_report(y_test, y_pred, output_dict=True),
    'feature_dimensions': X_train.shape[1]
}

# Add AUC-ROC if calculated
if 'auc_roc' in locals():
    metadata['performance_metrics']['test_auc_roc'] = float(auc_roc)

# Save metadata
with open(f'saved_models/{version_str}_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

# Also save as JSON for readability
import json
with open(f'saved_models/{version_str}_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False, default=str)

# 5.4 Save prediction examples
print("Saving prediction examples...")
predictions_df = pd.DataFrame({
    'sentence': df.loc[y_test.index, 'Sentence'].reset_index(drop=True),
    'true_label': y_test.values,
    'predicted_label': y_pred,
    'confidence': np.max(y_pred_proba, axis=1)
})

# Add class probabilities for each class
for i, class_name in enumerate(sorted(np.unique(y_train_full))):
    predictions_df[f'prob_class_{class_name}'] = y_pred_proba[:, i]

predictions_df.to_csv(f'saved_models/{version_str}_predictions.csv',
                      index=False, encoding='utf-8-sig')

# ============================================
# 6. CREATE LOADING SCRIPT
# ============================================
print("\n" + "="*50)
print("CREATING LOADING UTILITY")
print("="*50)

# Create a simple loading script
loading_script = f'''"""
Loading script for model: {version_str}
"""
import joblib
import pickle
from sentence_transformers import SentenceTransformer

def load_model_pipeline(model_dir='saved_models/{version_str}'):
    """
    Load the complete model pipeline
    """
    # Load metadata
    with open(f'{{model_dir}}_metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)

    # Load classifier
    classifier = joblib.load(f'{{model_dir}}_classifier.pkl')

    # Load sentence transformer
    sentence_model = SentenceTransformer(f'{{model_dir}}_sentence_transformer')

    return {{
        'sentence_model': sentence_model,
        'classifier': classifier,
        'metadata': metadata
    }}

def predict(texts, model_pipeline):
    """
    Make predictions on new texts
    """
    # Encode texts
    embeddings = model_pipeline['sentence_model'].encode(texts)

    # Make predictions
    predictions = model_pipeline['classifier'].predict(embeddings)
    probabilities = model_pipeline['classifier'].predict_proba(embeddings)

    return predictions, probabilities

# Example usage:
# model_pipeline = load_model_pipeline()
# texts = ["Exemple de texte en arabe", "Autre exemple"]
# preds, probs = predict(texts, model_pipeline)
'''

with open(f'saved_models/load_model_{version_str}.py', 'w', encoding='utf-8') as f:
    f.write(loading_script)

# ============================================
# 7. SUMMARY
# ============================================
print("\n" + "="*50)
print("TRAINING COMPLETE - SUMMARY")
print("="*50)
print(f"Model version: {version_str}")
print(f"Test Accuracy: {accuracy:.3f}")
print(f"Best F1-macro (CV): {grid_search.best_score_:.3f}")
print(f"\nSaved files in 'saved_models/':")
print(f"  • {version_str}_sentence_transformer/ (folder)")
print(f"  • {version_str}_classifier.pkl")
print(f"  • {version_str}_metadata.pkl")
print(f"  • {version_str}_metadata.json")
print(f"  • {version_str}_predictions.csv")
print(f"  • load_model_{version_str}.py")
print(f"  • confusion_matrix.png")
print(f"  • misclassified_examples.csv")

# Save final summary
summary = f"""
MODEL TRAINING SUMMARY
=====================
Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Model ID: {version_str}

DATA STATISTICS:
- Total samples: {len(df)}
- Training samples: {len(X_train_full)}
- Test samples: {len(X_test)}
- Number of classes: {len(np.unique(y_train_full))}

PERFORMANCE:
- Test Accuracy: {accuracy:.3f}
- Best CV F1-macro: {grid_search.best_score_:.3f}
- Error Rate: {error_rate:.2%}

HYPERPARAMETERS:
{grid_search.best_params_}

FILES SAVED:
1. Model files in 'saved_models/{version_str}_*'
2. Visualization: confusion_matrix.png
3. Error analysis: misclassified_examples.csv
4. Loading script: load_model_{version_str}.py
"""

print(summary)

# Save summary to file
with open(f'saved_models/{version_str}_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("\n" + "="*50)
print("All models and metadata saved successfully!")
print("Use the loading script to reload the model for inference.")
print("="*50)

In [ ]:
####best one suite---test sentences
import joblib
from sentence_transformers import SentenceTransformer

# =========================
# Load models
# =========================
clf = joblib.load("/content/stance_classifier_logreg.pkl")

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Optional label mapping (edit if needed)
label_map = {
    0: "AGAINST",
    1: "NEUTRAL",
    2: "FAVOR"
}

print("✅ Model loaded successfully!")
print("Type a sentence to predict its stance.")
print("Type 'exit' to stop.\n")

# =========================
# Interactive loop
# =========================
while True:
    sentence = input("📝 Enter your sentence: ")

    if sentence.lower() == "exit":
        print("👋 Exiting...")
        break

    # Encode sentence
    embedding = model.encode([sentence])

    # Predict
    pred = clf.predict(embedding)[0]
    prob = clf.predict_proba(embedding)[0]

    print(f"➡ Predicted label: {label_map.get(pred, pred)}")
    print(f"➡ Confidence: {prob.max():.3f}")
    print("-" * 50)


In [ ]:
####best one 2----fill with labels
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer

# =========================
# Load your trained models
# =========================
clf = joblib.load("/content/stance_classifier_logreg.pkl")
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Models loaded")

# =========================
# Load validation file (Arabic)
# =========================
input_path = "/content/Subtask_B_val_noLabel (2).csv"
df_val = pd.read_csv(input_path, encoding="utf-8")

print("✅ Validation file loaded")
print(df_val.head())

# =========================
# Encode sentences
# =========================
sentences = df_val["sentence"].astype(str).tolist()
embeddings = encoder.encode(sentences, show_progress_bar=True)

# =========================
# Predict labels
# =========================
predictions = clf.predict(embeddings)

# =========================
# Fill the label column
# =========================
df_val["label"] = predictions

# =========================
# Save filled CSV
# =========================
##### here the file:  Subtask_B_val_noLabel=== Subtask_B_val_filled.csv
output_path = "/content/Subtask_B_val_filled.csv"
df_val.to_csv(output_path, index=False, encoding="utf-8-sig")  # UTF-8 with BOM for Excel compatibility

print(f"\n✅ Submission file ready: {output_path}")


In [ ]:
# interactive_test.py
# interactive_test.py
import joblib
import sys
import os

def interactive_test(model_path):
    # Load model
    model_data = joblib.load(model_path)
    model = model_data['model']
    embedder = model_data['sentence_transformer']

    print("🎯 Arabic Stance Detection Interactive Test")
    print("Type Arabic sentences to test (or 'quit' to exit)")
    print("-" * 50)

    while True:
        # Get user input
        text = input("\n📝 Enter Arabic sentence: ").strip()

        if text.lower() in ['quit', 'exit', 'خروج']:
            print("Goodbye!")
            break

        if not text:
            continue

        # Predict
        embedding = embedder.encode([text])
        prediction = model.predict(embedding)[0]
        probabilities = model.predict_proba(embedding)[0]

        # Display results
        print(f"\n🔍 Results for: '{text}'")
        print(f"   Prediction: {prediction}")

        # Show all probabilities
        print(f"   Probabilities:")
        classes = model_data['classes']
        for cls, prob in zip(classes, probabilities):
            print(f"     • {cls}: {prob:.2%}")

        # Show confidence
        confidence = max(probabilities)
        print(f"   Confidence: {confidence:.2%}")

        # Interpretation
        if confidence > 0.8:
            print("   💡 Model is confident")
        elif confidence > 0.6:
            print("   🤔 Model is somewhat confident")
        else:
            print("   ⚠️  Model is uncertain")

# Run interactive test
if __name__ == "__main__":
    # Check command line arguments
    if len(sys.argv) > 1 and not sys.argv[1].startswith('-'):
        # First argument that doesn't start with - is the model file
        model_file = sys.argv[1]
    else:
        # No filename provided, look for available models
        print("No model file specified. Looking for available models...")

        # List available model files
        joblib_files = [f for f in os.listdir('.') if f.endswith('.joblib')]
        pkl_files = [f for f in os.listdir('.') if f.endswith('.pkl')]

        all_files = joblib_files + pkl_files

        if not all_files:
            print("No model files found in current directory.")
            sys.exit(1)

        # Show available files
        print("\nAvailable model files:")
        for i, file in enumerate(all_files, 1):
            print(f"  {i}. {file}")

        # Ask user to choose
        try:
            choice = input(f"\nSelect model file (1-{len(all_files)}) or enter filename: ")
            if choice.isdigit():
                model_file = all_files[int(choice)-1]
            else:
                model_file = choice
        except:
            # Default to the first .joblib file if available
            if joblib_files:
                model_file = joblib_files[0]
                print(f"Using default: {model_file}")
            else:
                model_file = pkl_files[0]
                print(f"Using default: {model_file}")

    # Check if file exists
    if not os.path.exists(model_file):
        print(f"\nError: File '{model_file}' not found!")
        print("Checking with .joblib extension...")
        if not model_file.endswith('.joblib'):
            model_file_with_ext = model_file + '.joblib'
            if os.path.exists(model_file_with_ext):
                model_file = model_file_with_ext
                print(f"Found: {model_file}")
            else:
                print(f"File '{model_file_with_ext}' also not found.")
                sys.exit(1)

    try:
        print(f"\nLoading model: {model_file}")
        interactive_test(model_file)
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nTrying to load with different parameters...")

        # Try different loading methods
        try:
            # Try loading just the model without the wrapper
            with open(model_file, 'rb') as f:
                import pickle
                model = pickle.load(f)
                print("Loaded with pickle directly")
                # You would need to adapt the interactive_test function
        except:
            print("Could not load the model. Please check the file format.")

In [ ]:
# ===========================================
# IMPORTS ET INSTALLATION
# ===========================================
######!pip install sentence-transformers scikit-learn pandas -q

####import pandas as pd
#####import numpy as np
import joblib
#####import re
#####from sentence_transformers import SentenceTransformer
#####from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ===========================================
# 1. CHARGER ET ENTRÂINER LE MODÈLE
# ===========================================
print("🤖 PRÉPARATION DU MODÈLE POUR LA COMPÉTITION")
print("="*60)


In [ ]:
# Charger les données d'entraînement
print("📥 Chargement des données d'entraînement...")
train_df = pd.read_csv('/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_train.csv', encoding='utf-8')
print(f"✅ Données d'entraînement: {len(train_df)} échantillons")
print(f"📊 Distribution: {train_df['stance_label'].value_counts().to_dict()}")


In [ ]:
# Charger le modèle SentenceTransformer
print("🔄 Chargement du modèle SentenceTransformer...")
sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modèle SentenceTransformer chargé")


In [ ]:
# Encoder TOUTES les données d'entraînement
print("🔢 Encodage des textes d'entraînement...")
X_train_all = sentence_model.encode(train_df['Sentence'].tolist(), show_progress_bar=True)
y_train_all = train_df['stance_label'].tolist()


In [ ]:
# Entraîner le classifieur final sur TOUTES les données
print("🏋️ Entraînement du classifieur LogisticRegression...")
clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0,
    solver='lbfgs',
    multi_class='multinomial'
)
clf.fit(X_train_all, y_train_all)
print("✅ Classifieur entraîné sur toutes les données")


In [ ]:
# Sauvegarder le modèle pour usage futur
joblib.dump(clf, 'stance_classifier_logreg.pkl')
print("💾 Modèle sauvegardé: 'stance_classifier_logreg.pkl'")

In [ ]:
# ===========================================
# 2. CHARGER LES DONNÉES DE VALIDATION
# ===========================================
print("\n" + "="*60)
print("📋 CHARGEMENT DES DONNÉES DE VALIDATION")
print("="*60)

# Supposons que vous avez le fichier de validation
# Si non, créez un exemple de structure
validation_path = "/content/drive/MyDrive/Competition-StanceNakba 2026/Subtask_B_val_noLabel.csv"

try:
    val_df = pd.read_csv(validation_path, encoding='utf-8')
    print(f"✅ Fichier de validation chargé: {len(val_df)} échantillons")

except FileNotFoundError:
    print("⚠️ Fichier de validation non trouvé. Création d'un exemple...")
    # Créer un exemple de structure
    val_data = {
        'id': range(100),
        'Sentence': [
            "انا مع التطبيع مع اسرائيل",
            "التطبيع مع اسرائيل مرفوض",
            "اهلا وسهلا باخواننا السوريين",
            "اخرجو السوريين من جزيرة العرب",
            "سيتم إرجاع السوريين بالأردن على سوريا",
        ] * 20  # Répéter pour avoir 100 échantillons
    }
    val_df = pd.DataFrame(val_data)
    print(f"✅ Exemple créé: {len(val_df)} échantillons")


In [ ]:
# Afficher la structure
print(f"\n📊 Structure du fichier de validation:")
print(f"Colonnes: {val_df.columns.tolist()}")
print(f"Premières lignes:")
print(val_df.head())


In [ ]:
# Identifier la colonne de texte
text_column = 'Sentence' if 'Sentence' in val_df.columns else 'text' if 'text' in val_df.columns else val_df.columns[1]
print(f"\n🔍 Colonne texte identifiée: '{text_column}'")

In [ ]:
# ===========================================
# 3. FONCTION DE PRÉDICTION
# ===========================================
def predict_stance_sbert(texts, sentence_model, classifier, batch_size=32):
    """
    Prédit les stances pour une liste de textes
    """
    predictions = []

    # Traiter par batch pour gérer la mémoire
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Encoder le batch
        batch_embeddings = sentence_model.encode(batch, show_progress_bar=False)

        # Prédire
        batch_predictions = classifier.predict(batch_embeddings)
        predictions.extend(batch_predictions)

        # Afficher progression
        if (i // batch_size) % 10 == 0:
            print(f"  Progression: {min(i+batch_size, len(texts))}/{len(texts)}")

    return predictions


In [ ]:
# ===========================================
# 4. GÉNÉRER LES PRÉDICTIONS
# ===========================================
print("\n" + "="*60)
print("🎯 GÉNÉRATION DES PRÉDICTIONS")
print("="*60)

# Prétraitement simple
print("🧹 Prétraitement des textes...")
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@\w+|#\w+|http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

val_df['clean_text'] = val_df[text_column].apply(clean_text)


In [ ]:
# Générer les prédictions
print("🔮 Génération des prédictions avec SentenceTransformer...")
val_predictions = predict_stance_sbert(
    val_df['clean_text'].tolist(),
    sentence_model,
    clf,
    batch_size=64
)

In [ ]:
# Ajouter les prédictions au DataFrame
val_df['stance_label'] = val_predictions

print(f"\n✅ Prédictions générées: {len(val_predictions)}")

In [ ]:
# ===========================================
# 5. CRÉER LE FICHIER DE SOUMISSION
# ===========================================
print("\n" + "="*60)
print("📄 PRÉPARATION DU FICHIER DE SOUMISSION")
print("="*60)

# Créer le DataFrame de soumission selon le format attendu
# Format typique: id, stance_label

# Identifier la colonne ID
id_column = 'id' if 'id' in val_df.columns else val_df.columns[0]
print(f"🔢 Colonne ID identifiée: '{id_column}'")

# Créer le fichier de soumission
submission_df = pd.DataFrame({
    'id': val_df[id_column],
    'stance_label': val_df['stance_label']
})


In [ ]:
# Sauvegarder
output_file = "Subtask_B_val_predictions.csv"
submission_df.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Fichier de soumission généré: {output_file}")
print(f"📊 Distribution des prédictions:")
print(submission_df['stance_label'].value_counts())

In [ ]:
# ===========================================
# 6. VALIDATION INTERNE (OPTIONNEL)
# ===========================================
print("\n" + "="*60)
print("🧪 VALIDATION INTERNE (sur données d'entraînement)")
print("="*60)

# Pour vérifier la qualité, prédire sur un subset d'entraînement
sample_size = min(100, len(train_df))
sample_indices = np.random.choice(len(train_df), sample_size, replace=False)

sample_texts = train_df.iloc[sample_indices]['Sentence'].tolist()
sample_true = train_df.iloc[sample_indices]['stance_label'].tolist()

sample_preds = predict_stance_sbert(sample_texts, sentence_model, clf, batch_size=32)


In [ ]:
# Calculer l'accuracy
correct = sum(1 for true, pred in zip(sample_true, sample_preds) if true == pred)
accuracy = correct / sample_size

print(f"🔍 Validation sur {sample_size} échantillons d'entraînement:")
print(f"   Accuracy: {accuracy:.3f} ({correct}/{sample_size} corrects)")

In [ ]:
# Afficher quelques exemples
print(f"\n👁️ Exemples de prédictions:")
for i in range(min(5, sample_size)):
    print(f"  {i+1}. '{sample_texts[i][:50]}...'")
    print(f"     → Vérité: {sample_true[i]}, Prédit: {sample_preds[i]}")


In [ ]:
# ===========================================
# 7. CODE POUR CHARGER ET UTILISER LE MODÈLE SAUVEGARDÉ
# ===========================================
print("\n" + "="*60)
print("🔄 CODE POUR UTILISATION FUTURE")
print("="*60)

# Créer un script de prédiction réutilisable
prediction_script = """
# SCRIPT DE PRÉDICTION POUR LA COMPÉTITION
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer

def load_models():
    '''Charge les modèles sauvegardés'''
    sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    classifier = joblib.load('stance_classifier_logreg.pkl')
    return sentence_model, classifier

def predict_new_data(new_csv_path, output_path='predictions.csv'):
    '''Prédit les stances pour un nouveau fichier CSV'''
    # 1. Charger les données
    df = pd.read_csv(new_csv_path, encoding='utf-8')

    # 2. Identifier les colonnes
    text_col = 'Sentence' if 'Sentence' in df.columns else df.columns[1]
    id_col = 'id' if 'id' in df.columns else df.columns[0]

    # 3. Charger les modèles
    sentence_model, classifier = load_models()

    # 4. Encoder et prédire
    embeddings = sentence_model.encode(df[text_col].tolist(), show_progress_bar=True)
    predictions = classifier.predict(embeddings)

    # 5. Sauvegarder
    submission = pd.DataFrame({
        'id': df[id_col],
        'stance_label': predictions
    })
    submission.to_csv(output_path, index=False, encoding='utf-8')

    print(f"✅ Prédictions sauvegardées dans: {output_path}")
    return submission

# Exemple d'utilisation
# predictions = predict_new_data('Subtask_B_val_noLabel.csv', 'ma_soumission.csv')
"""

with open('prediction_script.py', 'w', encoding='utf-8') as f:
    f.write(prediction_script)

print("💾 Script sauvegardé: 'prediction_script.py'")

In [ ]:
# ===========================================
# 8. RÉSUMÉ FINAL
# ===========================================
print("\n" + "="*60)
print("🎉 PRÊT POUR LA COMPÉTITION !")
print("="*60)

print(f"""
📋 RÉSUMÉ:
├── Modèle: SentenceTransformer + LogisticRegression
├── Données d'entraînement: {len(train_df)} échantillons
├── Prédictions générées: {len(submission_df)} échantillons
├── Fichier de soumission: {output_file}
├── Distribution: {submission_df['stance_label'].value_counts().to_dict()}
└── Validation interne: {accuracy:.3f} accuracy

📁 FICHIERS GÉNÉRÉS:
1. {output_file} → À UPLOADER sur la plateforme
2. stance_classifier_logreg.pkl → Modèle sauvegardé
3. prediction_script.py → Script réutilisable

🚀 ÉTAPES FINALES:
1. Téléchargez '{output_file}' depuis Colab
2. Uploadez-le sur la plateforme de la compétition
3. Vérifiez le format requis (peut-être besoin de renommer les colonnes)

📊 PREMIÈRES LIGNES DE MA SOUMISSION:
""")

print(submission_df.head(10))
print("\n✅ Bonne chance pour la compétition !")